## DefectDB — Defect Structure Viewer
Browse the DFT defect dataset and visualize any relaxed defect structure.
The interactive 3D mode uses [MatterViz](https://github.com/janosh/matterviz): a 4-view grid
(front / side / top / isometric) with the defect site rendered in **black**.


In [ ]:
import os
import io
import json
import re
import zipfile
import requests
import base64
import tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
import time

# Widget Imports
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown

# Structure Tools
try:
    from pymatgen.core import Structure
    from pymatgen.io.cif import CifWriter
    from pymatgen.io.vasp import Poscar
    from ase.io import read
    from ase.visualize.plot import plot_atoms
    defectdb_studio_HAS_STRUC_TOOLS = True
except ImportError:
    defectdb_studio_HAS_STRUC_TOOLS = False

# Plotly for interactive 3D
try:
    import plotly.graph_objects as go
    defectdb_studio_HAS_PLOTLY = True
except ImportError:
    defectdb_studio_HAS_PLOTLY = False

# =============================================================================
# CONFIGURATION
# =============================================================================
defectdb_studio_THEORY_CONFIG = {
    'PBEsol': {
        'url': 'https://github.com/msehabibur/DefectDB/releases/download/DefectDB/DefectDB.zip',
        'csv_filename': 'cdsete_defect_library_generation_pbesol.csv',
        'cache_subdir': 'DefectDB_PBEsol',
        'available': True,
        'display_name': 'PBEsol'
    },
    'HSE+SOC': {
        'url': None,  # Will be added later
        'csv_filename': 'cdsete_defect_library_generation_hse_soc.csv',
        'cache_subdir': 'DefectDB_HSE_SOC',
        'available': False,
        'display_name': 'HSE06+SOC'
    }
}

defectdb_studio_CACHE_DIR = Path.home() / "DefectDB_cache"
defectdb_studio_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# UTILITIES
# =============================================================================

def defectdb_studio_create_download_link(defectdb_content, defectdb_filename, defectdb_content_type='text/plain'):
    """Create auto-download link"""
    if isinstance(defectdb_content, str):
        defectdb_content = defectdb_content.encode()
    
    defectdb_b64 = base64.b64encode(defectdb_content).decode()
    defectdb_href = f'data:{defectdb_content_type};base64,{defectdb_b64}'
    
    defectdb_safe_id = defectdb_filename.replace(".", "_").replace(" ", "_") + f"_{int(time.time()*1000)}".replace("+", "_")
    defectdb_html = f'''
    <a href="{defectdb_href}" 
       download="{defectdb_filename}" 
       style="display:none" 
       id="dl_{defectdb_safe_id}"></a>
    <script>
        document.getElementById("dl_{defectdb_safe_id}").click();
    </script>
    '''
    return HTML(defectdb_html)

def defectdb_studio_download_database(defectdb_theory_level, defectdb_progress_widget=None):
    """Download and extract DefectDB from GitHub with extended silent retry logic"""
    defectdb_config = defectdb_studio_THEORY_CONFIG[defectdb_theory_level]
    
    if not defectdb_config['available'] or defectdb_config['url'] is None:
        raise ValueError(f"{defectdb_theory_level} dataset is not available yet")
    
    defectdb_cache_subdir = defectdb_studio_CACHE_DIR / defectdb_config['cache_subdir']
    defectdb_zip_path = defectdb_cache_subdir / "DefectDB.zip"
    defectdb_extract_path = defectdb_cache_subdir / "DefectDB"
    
    # Create cache directory
    defectdb_cache_subdir.mkdir(parents=True, exist_ok=True)
    
    # Check if already downloaded
    if defectdb_extract_path.exists() and (defectdb_extract_path / defectdb_config['csv_filename']).exists():
        if defectdb_progress_widget:
            defectdb_progress_widget.value = 100
            defectdb_progress_widget.description = "Cached:"
        return defectdb_extract_path
    
    # Download with progress and extended retry logic - SILENT ERROR HANDLING
    if defectdb_progress_widget:
        defectdb_progress_widget.description = "Downloading:"
    
    defectdb_max_retries = 10  # Increased retries
    defectdb_retry_delay = 3   # Longer delay between retries
    
    for defectdb_attempt in range(defectdb_max_retries):
        try:
            # Download with extended timeout and SSL verification
            defectdb_session = requests.Session()
            defectdb_session.headers.update({
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            })
            
            defectdb_response = defectdb_session.get(
                defectdb_config['url'],
                stream=True,
                timeout=(30, 60),  # Extended timeouts: (30s connection, 60s read)
                verify=True,
                allow_redirects=True
            )
            defectdb_response.raise_for_status()
            
            defectdb_total_size = int(defectdb_response.headers.get('content-length', 0))
            
            with open(defectdb_zip_path, 'wb') as defectdb_f:
                defectdb_downloaded = 0
                defectdb_update_threshold = max(defectdb_total_size // 20, 5 * 1024 * 1024)
                defectdb_last_update = 0
                
                for defectdb_chunk in defectdb_response.iter_content(chunk_size=8192):
                    if defectdb_chunk:
                        defectdb_f.write(defectdb_chunk)
                        defectdb_downloaded += len(defectdb_chunk)
                        
                        if defectdb_downloaded - defectdb_last_update >= defectdb_update_threshold or defectdb_downloaded == defectdb_total_size:
                            if defectdb_progress_widget and defectdb_total_size > 0:
                                defectdb_progress_widget.value = int((defectdb_downloaded / defectdb_total_size) * 100)
                            defectdb_last_update = defectdb_downloaded
            
            # Successfully downloaded, break retry loop
            break
            
        except (requests.exceptions.RequestException, TimeoutError, ConnectionError) as defectdb_e:
            # SILENT ERROR HANDLING - Don't show error messages
            if defectdb_attempt < defectdb_max_retries - 1:
                if defectdb_progress_widget:
                    defectdb_progress_widget.description = f"Retrying {defectdb_attempt+1}:"
                    defectdb_progress_widget.value = 0
                time.sleep(defectdb_retry_delay * (defectdb_attempt + 1))
            else:
                # After all retries exhausted, silently return None
                if defectdb_progress_widget:
                    defectdb_progress_widget.description = "Failed:"
                    defectdb_progress_widget.value = 0
                return None
    
    # Extract
    if defectdb_progress_widget:
        defectdb_progress_widget.description = "Extracting:"
        defectdb_progress_widget.value = 0
    
    try:
        with zipfile.ZipFile(defectdb_zip_path, 'r') as defectdb_zip_ref:
            defectdb_members = defectdb_zip_ref.namelist()
            defectdb_total_members = len(defectdb_members)
            
            for defectdb_i, defectdb_member in enumerate(defectdb_members):
                defectdb_zip_ref.extract(defectdb_member, defectdb_cache_subdir)
                if defectdb_progress_widget and defectdb_i % max(1, defectdb_total_members // 20) == 0:
                    defectdb_progress_widget.value = int((defectdb_i / defectdb_total_members) * 100)
    except Exception as defectdb_e:
        # Silent error handling for extraction
        if defectdb_progress_widget:
            defectdb_progress_widget.description = "Failed:"
            defectdb_progress_widget.value = 0
        return None
    
    if defectdb_progress_widget:
        defectdb_progress_widget.value = 100
        defectdb_progress_widget.description = "Complete:"
    
    return defectdb_extract_path

def defectdb_studio_load_csv_data(defectdb_theory_level):
    """Load CSV database for specific theory level"""
    defectdb_config = defectdb_studio_THEORY_CONFIG[defectdb_theory_level]
    defectdb_cache_subdir = defectdb_studio_CACHE_DIR / defectdb_config['cache_subdir']
    defectdb_db_path = defectdb_cache_subdir / "DefectDB" / defectdb_config['csv_filename']
    
    if not defectdb_db_path.exists():
        return None
    return pd.read_csv(defectdb_db_path)

def defectdb_studio_index_structures(defectdb_theory_level):
    """Index all structure files by compound/defect/charge for specific theory level"""
    defectdb_config = defectdb_studio_THEORY_CONFIG[defectdb_theory_level]
    defectdb_cache_subdir = defectdb_studio_CACHE_DIR / defectdb_config['cache_subdir']
    defectdb_db_path = defectdb_cache_subdir / "DefectDB"
    
    if not defectdb_db_path.exists():
        return {}
    
    defectdb_structures = {}
    
    for defectdb_comp_dir in defectdb_db_path.iterdir():
        if not defectdb_comp_dir.is_dir() or defectdb_comp_dir.name.startswith('.'):
            continue
        
        defectdb_comp_name = defectdb_comp_dir.name
        defectdb_structures[defectdb_comp_name] = {}
        
        for defectdb_def_dir in defectdb_comp_dir.iterdir():
            if not defectdb_def_dir.is_dir() or defectdb_def_dir.name.lower() == 'bulk':
                continue
            
            defectdb_def_name = defectdb_def_dir.name
            defectdb_structures[defectdb_comp_name][defectdb_def_name] = {}
            
            for defectdb_chg_dir in defectdb_def_dir.iterdir():
                if not defectdb_chg_dir.is_dir():
                    continue
                
                defectdb_chg_name = defectdb_chg_dir.name
                
                defectdb_priority = ["CONTCAR", "POSCAR", "Relaxed.cif", "Final.cif", "structure.cif"]
                for defectdb_fname in defectdb_priority:
                    defectdb_fpath = defectdb_chg_dir / defectdb_fname
                    if defectdb_fpath.exists():
                        defectdb_structures[defectdb_comp_name][defectdb_def_name][defectdb_chg_name] = defectdb_fpath
                        break
    
    return defectdb_structures

def defectdb_studio_check_dataset_exists(defectdb_theory_level):
    """Check if dataset for theory level is already downloaded"""
    defectdb_config = defectdb_studio_THEORY_CONFIG[defectdb_theory_level]
    defectdb_cache_subdir = defectdb_studio_CACHE_DIR / defectdb_config['cache_subdir']
    defectdb_db_path = defectdb_cache_subdir / "DefectDB" / defectdb_config['csv_filename']
    return defectdb_db_path.exists()

def defectdb_studio_format_defect(defectdb_defect_str):
    """Format defect string with proper subscripts: V_Cd → V$_{Cd}$"""
    defectdb_parts = defectdb_defect_str.split('+')
    defectdb_formatted_parts = []
    
    for defectdb_part in defectdb_parts:
        defectdb_part = defectdb_part.strip()
        
        if defectdb_part.endswith('_i'):
            defectdb_elem = defectdb_part[:-2]
            defectdb_formatted_parts.append(f"{defectdb_elem}$_{{i}}$")
        elif '_' in defectdb_part:
            defectdb_left, defectdb_right = defectdb_part.split('_', 1)
            defectdb_formatted_parts.append(f"{defectdb_left}$_{{{defectdb_right}}}$")
        else:
            defectdb_formatted_parts.append(defectdb_part)
    
    return '+'.join(defectdb_formatted_parts)

def defectdb_studio_format_compound(defectdb_comp_str):
    """Format compound string with subscripts: Cd0.5Zn0.5Te → Cd$_{0.5}$Zn$_{0.5}$Te"""
    defectdb_result = re.sub(r'([A-Z][a-z]?)(\d+\.?\d*)', r'\1$_{\2}$', defectdb_comp_str)
    return defectdb_result

def defectdb_studio_force_float(defectdb_val):
    """Safely convert to float"""
    try:
        if defectdb_val is None: return np.nan
        if isinstance(defectdb_val, str):
            defectdb_val = defectdb_val.strip()
            if defectdb_val == '' or defectdb_val.lower() == 'nan': return np.nan
        return float(defectdb_val)
    except (ValueError, TypeError):
        return np.nan

def defectdb_studio_generate_formation_energy_plot(defectdb_df, defectdb_compound, defectdb_chem_pot_label, defectdb_chem_pot_col, defectdb_defects_to_plot):
    """Generate formation energy plot with proper formatting - 6x6 SIZE"""
    plt.close('all')
    defectdb_fig, defectdb_ax = plt.subplots(figsize=(6, 6), facecolor='white')
    
    if defectdb_df.empty:
        return defectdb_fig
    
    defectdb_row0 = defectdb_df.iloc[0]
    defectdb_gap = defectdb_studio_force_float(defectdb_row0.get('gap', 1.5))
    if np.isnan(defectdb_gap):
        defectdb_gap = 1.5
    
    defectdb_formatted_compound = defectdb_studio_format_compound(defectdb_compound)
    defectdb_ax.set_title(rf"{defectdb_formatted_compound} ($\mu$={defectdb_chem_pot_label})", fontsize=22)
    defectdb_ax.set_xlabel("Fermi Level (eV)", fontsize=18)
    defectdb_ax.set_ylabel("Formation Energy (eV)", fontsize=18)
    defectdb_ax.axvline(0, ls='--', c='gray', lw=2, alpha=0.7)
    defectdb_ax.axvline(defectdb_gap, ls='--', c='gray', lw=2, alpha=0.7)
    defectdb_ax.fill_between([0, defectdb_gap], -10, 10, color='#f5f5f5', alpha=0.3)
    defectdb_ax.set_xlim(-0.3, defectdb_gap+0.3)
    defectdb_ax.tick_params(labelsize=18)
    defectdb_ax.grid(True, alpha=0.2, linestyle=':', linewidth=1)
    
    defectdb_colors = plt.cm.tab10.colors
    defectdb_EF = np.linspace(-0.5, defectdb_gap+0.5, 200)
    defectdb_y_max_track = []
    
    for defectdb_idx, defectdb_defect in enumerate(defectdb_defects_to_plot):
        defectdb_d_data = defectdb_df[defectdb_df['Defect'] == defectdb_defect]
        if defectdb_d_data.empty:
            continue
        
        defectdb_r = defectdb_d_data.iloc[0]
        defectdb_mu = defectdb_studio_force_float(defectdb_r.get(defectdb_chem_pot_col, 0))
        defectdb_vbm = defectdb_studio_force_float(defectdb_r.get('VBM', 0))
        defectdb_pure = defectdb_studio_force_float(defectdb_r.get('Toten_pure', 0))
        
        defectdb_energies = []
        for defectdb_q in [2, 1, 0, -1, -2]:
            defectdb_t_col = f"Toten_{'p' if defectdb_q>0 else 'm' if defectdb_q<0 else 'neut'}{abs(defectdb_q) if defectdb_q!=0 else ''}"
            defectdb_c_col = f"Corr_{'p' if defectdb_q>0 else 'm' if defectdb_q<0 else 'neut'}{abs(defectdb_q) if defectdb_q!=0 else ''}"
            
            defectdb_val_t = defectdb_studio_force_float(defectdb_r.get(defectdb_t_col))
            defectdb_val_c = defectdb_studio_force_float(defectdb_r.get(defectdb_c_col))
            
            if not np.isnan(defectdb_val_t) and not np.isnan(defectdb_val_c) and not np.isnan(defectdb_pure):
                defectdb_E_form = defectdb_val_t - defectdb_pure + defectdb_mu + defectdb_q*(defectdb_EF + defectdb_vbm) + defectdb_val_c
                defectdb_energies.append(defectdb_E_form)
        
        if defectdb_energies:
            defectdb_min_energy = np.min(defectdb_energies, axis=0)
            defectdb_formatted_defect = defectdb_studio_format_defect(defectdb_defect)
            defectdb_ax.plot(defectdb_EF, defectdb_min_energy, lw=4, label=defectdb_formatted_defect, color=defectdb_colors[defectdb_idx % len(defectdb_colors)])
            
            defectdb_valid_y = defectdb_min_energy[(defectdb_EF>=0) & (defectdb_EF<=defectdb_gap)]
            defectdb_clean_y = [defectdb_y for defectdb_y in defectdb_valid_y if np.isfinite(defectdb_y)]
            
            if len(defectdb_clean_y) > 0:
                defectdb_y_max_track.append(np.max(defectdb_clean_y))
    
    defectdb_y_max_track = [defectdb_y for defectdb_y in defectdb_y_max_track if np.isfinite(defectdb_y)]
    
    if defectdb_y_max_track:
        defectdb_ax.set_ylim(0, max(defectdb_y_max_track)+1.0)
    else:
        defectdb_ax.set_ylim(0, 4)
    
    defectdb_ax.legend(loc='upper left', fontsize=16, framealpha=0.95, shadow=True)
    
    defectdb_fig.subplots_adjust(left=0.15, right=0.95, top=0.93, bottom=0.12)
    
    return defectdb_fig

# =============================================================================
# MatterViz interactive 3D viewer (https://github.com/janosh/matterviz)
# Assets are self-hosted in this repo under matterviz/ and served via jsDelivr.
# =============================================================================
defectdb_studio_MATTERVIZ_BASE = os.environ.get(
    "DEFECTDB_MATTERVIZ_BASE",
    "https://cdn.jsdelivr.net/gh/msehabibur/DefectDB@main/matterviz")

def defectdb_studio_visualize_structure_matterviz(defectdb_struct_path, defectdb_title):
    """Interactive 3D structure rendered with MatterViz (4-view grid, defect species in black)"""
    import uuid as defectdb_uuid
    from IPython.display import HTML as defectdb_HTML, Javascript as defectdb_JS, display as defectdb_display
    defectdb_struct = Structure.from_file(str(defectdb_struct_path))
    with tempfile.NamedTemporaryFile(suffix='.cif', delete=False, mode='w') as defectdb_tmp:
        CifWriter(defectdb_struct).write_file(defectdb_tmp.name)
        defectdb_atoms = read(defectdb_tmp.name)
    defectdb_defect_name = Path(defectdb_struct_path).parent.parent.name
    defectdb_host_elems = {'Cd', 'Zn', 'Te', 'Se', 'S'}
    defectdb_defect_elems = []
    for defectdb_tok in defectdb_defect_name.split('+'):
        defectdb_head = defectdb_tok.split('_')[0].strip()
        if defectdb_head and defectdb_head != 'V' and defectdb_head not in defectdb_host_elems:
            defectdb_defect_elems.append(defectdb_head)
    defectdb_sjson = json.dumps(defectdb_struct.as_dict())
    defectdb_div_id = f"mv-{defectdb_uuid.uuid4().hex[:10]}"
    defectdb_html_part = (
        '<div style="border:1px solid #e5e7eb;border-radius:8px;overflow:hidden;">'
        f'<div style="padding:6px 10px;font-weight:600;color:#4b5563;background:#f9fafb;">{defectdb_title} &mdash; MatterViz</div>'
        f'<div id="{defectdb_div_id}" style="width:100%;height:900px;background:#ffffff;"></div></div>')
    defectdb_js_part = f"""
(function() {{
  var mvBase = "{defectdb_studio_MATTERVIZ_BASE}";
  if (!document.getElementById('mv-css')) {{
    var l = document.createElement('link');
    l.id = 'mv-css'; l.rel = 'stylesheet'; l.href = mvBase + '/mv-asset-index.css';
    document.head.appendChild(l);
  }}
  var mvStructure = {defectdb_sjson};
  var mvTryRender = function(mvAttempt) {{
    var mvEl = document.getElementById('{defectdb_div_id}');
    if (!mvEl) {{
      if (mvAttempt < 50) setTimeout(function() {{ mvTryRender(mvAttempt + 1); }}, 200);
      return;
    }}
    import(mvBase + '/mv-app.js').then(function() {{
      window.renderMatterVizGrid(mvStructure, mvEl,
        {{ defect_elems: {json.dumps(defectdb_defect_elems)}, grid: true }});
    }});
  }};
  mvTryRender(0);
}})();
"""

    class defectdb_MVDisplay:
        def __init__(self, defectdb_h, defectdb_j):
            self.defectdb_h = defectdb_h
            self.defectdb_j = defectdb_j
        def _ipython_display_(self):
            defectdb_display(defectdb_HTML(self.defectdb_h))
            defectdb_display(defectdb_JS(self.defectdb_j))

    return defectdb_MVDisplay(defectdb_html_part, defectdb_js_part), defectdb_atoms

def defectdb_studio_visualize_structure_static(defectdb_struct_path, defectdb_title):
    """Create compact 2x2 static structure visualization"""
    if not defectdb_studio_HAS_STRUC_TOOLS:
        return None
    
    # Load structure
    if defectdb_struct_path.suffix == '.cif':
        defectdb_atoms = read(str(defectdb_struct_path))
    else:
        defectdb_struct = Structure.from_file(str(defectdb_struct_path))
        with tempfile.NamedTemporaryFile(suffix='.cif', delete=False, mode='w') as defectdb_tmp:
            CifWriter(defectdb_struct).write_file(defectdb_tmp.name)
            defectdb_atoms = read(defectdb_tmp.name)
    
    defectdb_fig = plt.figure(figsize=(8, 6.5), facecolor='white')
    defectdb_gs = GridSpec(2, 2, figure=defectdb_fig, hspace=0.25, wspace=0.15)
    
    defectdb_views = [
        ('Front (100)', '0x,0y,0z'),
        ('Side (010)', '0x,90y,0z'),
        ('Top (001)', '90x,0y,0z'),
        ('Isometric 3D', '45x,30y,15z')
    ]
    
    for defectdb_idx, (defectdb_view_name, defectdb_rotation) in enumerate(defectdb_views):
        defectdb_ax = defectdb_fig.add_subplot(defectdb_gs[defectdb_idx // 2, defectdb_idx % 2])
        plot_atoms(defectdb_atoms, defectdb_ax, rotation=defectdb_rotation, show_unit_cell=2, radii=0.5)
        defectdb_ax.set_title(defectdb_view_name, fontsize=11, pad=6)
        defectdb_ax.axis('off')
    
    defectdb_fig.suptitle(defectdb_title, fontsize=14, y=0.98)
    
    defectdb_fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.02, hspace=0.25, wspace=0.15)
    
    return defectdb_fig, defectdb_atoms

def defectdb_studio_visualize_structure_interactive(defectdb_struct_path, defectdb_title):
    """Create interactive 3D structure with Plotly"""
    if not defectdb_studio_HAS_PLOTLY or not defectdb_studio_HAS_STRUC_TOOLS:
        return None
    
    # Load structure
    if defectdb_struct_path.suffix == '.cif':
        defectdb_atoms = read(str(defectdb_struct_path))
    else:
        defectdb_struct = Structure.from_file(str(defectdb_struct_path))
        with tempfile.NamedTemporaryFile(suffix='.cif', delete=False, mode='w') as defectdb_tmp:
            CifWriter(defectdb_struct).write_file(defectdb_tmp.name)
            defectdb_atoms = read(defectdb_tmp.name)
    
    defectdb_positions = defectdb_atoms.get_positions()
    defectdb_symbols = defectdb_atoms.get_chemical_symbols()
    
    defectdb_color_map = {
        'Cd': '#FFD700', 'Zn': '#7B68EE', 'Te': '#FF6347', 'Se': '#FFA500',
        'Cu': '#FF8C00', 'Ag': '#C0C0C0', 'Au': '#FFD700', 'Cl': '#00FF00',
        'As': '#9370DB', 'Sb': '#DDA0DD', 'P': '#FFA500', 'S': '#FFFF00'
    }
    
    defectdb_size_map = {
        'Cd': 12, 'Zn': 10, 'Te': 11, 'Se': 10,
        'Cu': 10, 'Ag': 11, 'Au': 11, 'Cl': 9,
        'As': 10, 'Sb': 11, 'P': 9, 'S': 9
    }
    
    defectdb_colors = [defectdb_color_map.get(defectdb_s, '#808080') for defectdb_s in defectdb_symbols]
    defectdb_sizes = [defectdb_size_map.get(defectdb_s, 10) for defectdb_s in defectdb_symbols]
    
    defectdb_fig = go.Figure(data=[go.Scatter3d(
        x=defectdb_positions[:, 0],
        y=defectdb_positions[:, 1],
        z=defectdb_positions[:, 2],
        mode='markers',
        marker=dict(
            size=defectdb_sizes,
            color=defectdb_colors,
            line=dict(color='black', width=1),
            opacity=0.9
        ),
        text=[f"{defectdb_s} (#{defectdb_i})" for defectdb_i, defectdb_s in enumerate(defectdb_symbols)],
        hoverinfo='text'
    )])
    
    # Add unit cell edges
    defectdb_cell = defectdb_atoms.get_cell()
    defectdb_cell_corners = [
        [0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0], [0, 0, 0],
        [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1], [0, 0, 1],
        [0, 1, 1], [0, 1, 0], [1, 1, 0], [1, 1, 1], [1, 0, 1], [1, 0, 0]
    ]
    
    defectdb_cell_positions = np.array([defectdb_cell[0] * defectdb_c[0] + defectdb_cell[1] * defectdb_c[1] + defectdb_cell[2] * defectdb_c[2] for defectdb_c in defectdb_cell_corners])
    
    defectdb_fig.add_trace(go.Scatter3d(
        x=defectdb_cell_positions[:, 0],
        y=defectdb_cell_positions[:, 1],
        z=defectdb_cell_positions[:, 2],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    defectdb_fig.update_layout(
        title=defectdb_title,
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=500,
        hovermode='closest'
    )
    
    return defectdb_fig, defectdb_atoms

# =============================================================================
# GUI APPLICATION
# =============================================================================

class DefectDBStudioApp:
    def __init__(self):
        self.defectdb_current_theory = 'PBEsol'
        self.defectdb_df = None
        self.defectdb_structures_index = {}
        self.defectdb_current_structure_path = None
        self.defectdb_current_structure_atoms = None
        self.defectdb_current_plot_fig = None
        self.defectdb_current_struct_fig = None
        
        self.defectdb_setup_widgets()
    
    def defectdb_setup_widgets(self):
        """Setup all widgets with light gray styling"""
        
        # Header - DARK GRAY THEME
        self.defectdb_header = widgets.HTML("""
        <style>
            .defectdb-header {
                background: linear-gradient(135deg, #9ca3af 0%, #6b7280 100%);
                padding: 30px;
                border-radius: 15px;
                box-shadow: 0 10px 30px rgba(0,0,0,0.2);
                margin-bottom: 20px;
            }
            .defectdb-title {
                color: white;
                font-size: 36px;
                font-weight: 900;
                margin: 0;
                text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
                text-align: center;
            }
            .defectdb-subtitle {
                color: #e5e7eb;
                font-size: 18px;
                margin-top: 8px;
                text-align: center;
            }
        </style>
        <div class="defectdb-header">
            <div class="defectdb-title">Module 1 → Access DFT Defect Dataset</div>
        </div>
        """)

        # Methodology Section
        self.defectdb_methodology = widgets.HTML("""
        <style>
            .defectdb-methodology {
                background: #f3f4f6;
                padding: 20px 25px;
                border-radius: 12px;
                border-left: 5px solid #6b7280;
                margin: 15px 0;
                box-shadow: 0 2px 8px rgba(0,0,0,0.1);
            }
            .defectdb-method-title {
                color: #374151;
                font-size: 20px;
                font-weight: 700;
                margin: 0 0 15px 0;
                border-bottom: 2px solid #9ca3af;
                padding-bottom: 8px;
            }
            .defectdb-method-subsection {
                color: #4b5563;
                font-size: 16px;
                font-weight: 600;
                margin: 12px 0 8px 0;
            }
            .defectdb-method-text {
                color: #6b7280;
                font-size: 14px;
                line-height: 1.7;
                margin: 8px 0;
                text-align: justify;
            }
            .defectdb-method-list {
                color: #6b7280;
                font-size: 14px;
                line-height: 1.6;
                margin: 5px 0 5px 20px;
            }
        </style>
        <div class="defectdb-methodology">
            <div class="defectdb-method-title">🔬 DFT Methodology and Computational Details</div>

            <div class="defectdb-method-subsection">Computational Approach</div>
            <div class="defectdb-method-text">
                DFT computations were performed using 3×3×3 cubic zincblende supercells of CdTe and CdSe<sub>x</sub>Te<sub>1-x</sub>
                alloys (x = 0, 0.06, 0.12, 0.20 0.25, 0.50, 0.75, 1) with the special quasirandom structures (SQS) method. All native defects,
                impurities, and complexes were introduced in optimized supercells with multiple charge states (q = -2, -1, 0, +1, +2).
                Calculations were performed using VASP with projector augmented wave (PAW) pseudopotentials.
            </div>

            <div class="defectdb-method-subsection">DFT Settings</div>
            <div class="defectdb-method-text">
                <b>Functional:</b> PBEsol (GGA) for geometry optimization, providing accurate lattice parameters
                (CdTe: 6.50 Å vs. experimental 6.48 Å).<br>
                <b>Energy Cutoff:</b> 500 eV plane-wave basis<br>
                <b>Convergence:</b> Forces &lt; 0.05 eV/Å<br>
                <b>K-points:</b> Gamma-centered 2×2×2 Monkhorst-Pack mesh<br>
                <b>Hybrid DFT:</b> HSE06 with α = 0.31 mixing parameter and spin-orbit coupling (HSE+SOC),
                accurately reproducing experimental CdTe bandgap (1.49 eV vs. 1.5 eV experimental)
            </div>

            <div class="defectdb-method-subsection">Chemical Potential Calculations</div>
            <div class="defectdb-method-text">
                Chemical potentials of all native species (Cd, Se, Te) and impurities (As, Cu, O, Cl) were calculated
                to define thermodynamic stability limits. The calculations prevent decomposition to elemental forms and
                eliminate formation of competing phases (e.g., Cd<sub>3</sub>As<sub>2</sub>, As<sub>2</sub>Te<sub>3</sub>, CuO).
                Chemical potentials are referenced to the lowest energy elemental standard states from the Materials Project database,
                enabling accurate defect formation energy predictions under various growth conditions (Cd-rich, Te-rich, Se-rich).
            </div>
        </div>
        """)
        
        # Theory Level Selector
        self.defectdb_theory_selector = widgets.ToggleButtons(
            options=[
                ('PBEsol', 'PBEsol'),
                ('HSE06+SOC', 'HSE+SOC')
            ],
            value='PBEsol',
            description='Level of Theory:',
            disabled=False,
            button_style='',
            style={'description_width': '120px', 'button_width': '150px'}
        )
        self.defectdb_theory_selector.observe(self.defectdb_on_theory_change, names='value')
        
        # Database control
        self.defectdb_btn_download_db = widgets.Button(
            description="Download Database",
            button_style='',
            layout=widgets.Layout(width='200px', height='45px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'}
        )
        self.defectdb_btn_download_db.on_click(self.defectdb_on_download_database)
        
        self.defectdb_progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Ready:',
            bar_style='info',
            style={'bar_color': '#9ca3af'},
            layout=widgets.Layout(width='400px')
        )
        
        self.defectdb_status_label = widgets.HTML(
            "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
            "<b>Status:</b> Ready - Click 'Download Database' to begin</div>"
        )
        
        # Create tabs
        self.defectdb_create_data_tab()
        self.defectdb_create_plot_tab()
        self.defectdb_create_structure_tab()
        
        # Tab widget
        self.defectdb_tabs = widgets.Tab()
        self.defectdb_tabs.children = [self.defectdb_tab_data, self.defectdb_tab_plot, self.defectdb_tab_structures]
        self.defectdb_tabs.set_title(0, "📂 Data Explorer")
        self.defectdb_tabs.set_title(1, "📊 Formation Energy")
        self.defectdb_tabs.set_title(2, "🔬 Structure Viewer")
    
    def defectdb_create_data_tab(self):
        """Tab 1: Data Explorer"""
        self.defectdb_data_filter_comp = widgets.Dropdown(
            description="Compound:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='300px')
        )
        
        self.defectdb_data_filter_defect = widgets.Text(
            description="Defect Filter:",
            placeholder="e.g., V_Cd",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='300px')
        )
        
        self.defectdb_btn_filter_data = widgets.Button(
            description="🔍 Filter",
            button_style='',
            layout=widgets.Layout(width='120px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'}
        )
        self.defectdb_btn_filter_data.on_click(self.defectdb_on_filter_data)
        
        self.defectdb_btn_export_csv = widgets.Button(
            description="📥 Export CSV",
            button_style='',
            layout=widgets.Layout(width='120px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'}
        )
        self.defectdb_btn_export_csv.on_click(self.defectdb_on_export_csv)
        
        self.defectdb_data_output = widgets.Output(
            layout={'border': '2px solid #e2e8f0', 'border_radius': '8px', 'padding': '10px', 'height': '500px', 'overflow': 'auto'}
        )
        
        self.defectdb_tab_data = widgets.VBox([
            widgets.HTML("<h3 style='color:#6b7280;'>📂 Defect Database</h3>"),
            widgets.HBox([self.defectdb_data_filter_comp, self.defectdb_data_filter_defect, self.defectdb_btn_filter_data, self.defectdb_btn_export_csv]),
            self.defectdb_data_output
        ])
    
    def defectdb_create_plot_tab(self):
        """Tab 2: Formation Energy Plotter"""
        self.defectdb_plot_comp_dropdown = widgets.Dropdown(
            description="Compound:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='100%')
        )
        self.defectdb_plot_comp_dropdown.observe(self.defectdb_on_plot_comp_change, names='value')
        
        self.defectdb_plot_chempot = widgets.Dropdown(
            options=[('Cd-rich', 'mu_Cd_rich'), ('Te-rich', 'mu_Te_rich')],
            value='mu_Cd_rich',
            description="Chem Pot:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='100%')
        )
        
        self.defectdb_plot_defects = widgets.SelectMultiple(
            description="Defects:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='100%', height='200px')
        )
        
        self.defectdb_btn_generate_plot = widgets.Button(
            description="📊 Generate Plot",
            button_style='',
            layout=widgets.Layout(width='100%', height='45px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'}
        )
        self.defectdb_btn_generate_plot.on_click(self.defectdb_on_generate_plot)
        
        self.defectdb_btn_download_plot = widgets.Button(
            description="📥 Download Image",
            button_style='',
            layout=widgets.Layout(width='100%', height='40px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'},
            disabled=True
        )
        self.defectdb_btn_download_plot.on_click(self.defectdb_on_download_plot)
        
        self.defectdb_plot_output = widgets.Output(
            layout={'border': '2px solid #e2e8f0', 'border_radius': '8px', 'padding': '10px'}
        )
        
        defectdb_left_panel = widgets.VBox([
            widgets.HTML("<h4 style='color:#6b7280;'>Settings</h4>"),
            self.defectdb_plot_comp_dropdown,
            self.defectdb_plot_chempot,
            self.defectdb_plot_defects,
            self.defectdb_btn_generate_plot,
            self.defectdb_btn_download_plot
        ], layout=widgets.Layout(width='300px', padding='10px'))
        
        defectdb_right_panel = widgets.VBox([
            widgets.HTML("<h4 style='color:#6b7280;'>Formation Energy Plot</h4>"),
            self.defectdb_plot_output
        ], layout=widgets.Layout(width='calc(100% - 320px)', padding='10px'))
        
        self.defectdb_tab_plot = widgets.HBox([defectdb_left_panel, defectdb_right_panel])
    
    def defectdb_create_structure_tab(self):
        """Tab 3: Structure Viewer"""
        self.defectdb_struct_comp = widgets.Dropdown(
            description="Compound:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='250px')
        )
        self.defectdb_struct_comp.observe(self.defectdb_on_struct_comp_change, names='value')
        
        self.defectdb_struct_defect = widgets.Dropdown(
            description="Defect:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='250px'),
            disabled=True
        )
        self.defectdb_struct_defect.observe(self.defectdb_on_struct_defect_change, names='value')
        
        self.defectdb_struct_charge = widgets.Dropdown(
            description="Charge:",
            style={'description_width': '100px'},
            layout=widgets.Layout(width='200px'),
            disabled=True
        )
        
        self.defectdb_struct_view_mode = widgets.ToggleButtons(
            options=['Static Views', 'Interactive 3D (MatterViz)'],
            value='Static Views',
            description='Mode:',
            style={'description_width': '60px'}
        )
        
        self.defectdb_btn_visualize = widgets.Button(
            description="🔬 Visualize",
            button_style='',
            layout=widgets.Layout(width='150px', height='40px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'},
            disabled=True
        )
        self.defectdb_btn_visualize.on_click(self.defectdb_on_visualize_structure)
        
        self.defectdb_btn_download_structure = widgets.Button(
            description="📥 Download Structure (CIF)",
            layout=widgets.Layout(width='220px', height='40px'),
            style={'font_weight': 'bold', 'button_color': '#f3f4f6', 'text_color': '#374151'},
            disabled=True
        )
        self.defectdb_btn_download_structure.on_click(self.defectdb_on_download_structure)
        
        self.defectdb_struct_info = widgets.HTML()
        
        self.defectdb_struct_output = widgets.Output(
            layout={'border': '2px solid #e2e8f0', 'border_radius': '8px', 'padding': '10px'}
        )
        
        self.defectdb_tab_structures = widgets.VBox([
            widgets.HTML("<h3 style='color:#6b7280;'>🔬 Crystal Structure Viewer</h3>"),
            widgets.HBox([self.defectdb_struct_comp, self.defectdb_struct_defect, self.defectdb_struct_charge]),
            widgets.HBox([self.defectdb_struct_view_mode, self.defectdb_btn_visualize]),
            widgets.HBox([self.defectdb_btn_download_structure]),
            self.defectdb_struct_info,
            self.defectdb_struct_output
        ])
    
    def defectdb_display(self):
        """Display the app"""
        display(widgets.VBox([
            self.defectdb_header,
            self.defectdb_methodology,
            widgets.HBox([self.defectdb_theory_selector], layout=widgets.Layout(justify_content='center', margin='10px 0')),
            widgets.HBox([self.defectdb_btn_download_db], layout=widgets.Layout(justify_content='center')),
            widgets.HBox([self.defectdb_progress_bar], layout=widgets.Layout(justify_content='center')),
            self.defectdb_status_label,
            widgets.HTML("<hr style='border:1px solid #e2e8f0;margin:20px 0;'>"),
            self.defectdb_tabs
        ]))
    
    # Event Handlers
    
    def defectdb_on_theory_change(self, defectdb_change):
        """Handle theory level change"""
        defectdb_new_theory = defectdb_change.new
        self.defectdb_current_theory = defectdb_new_theory
        
        defectdb_config = defectdb_studio_THEORY_CONFIG[defectdb_new_theory]
        
        # Check if dataset is available
        if not defectdb_config['available']:
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
                f"<b>Status:</b> {defectdb_config['display_name']} dataset is coming soon! 🚀</div>"
            )
            self.defectdb_btn_download_db.disabled = True
            self.defectdb_df = None
            self.defectdb_structures_index = {}
            
            # Clear all outputs
            with self.defectdb_data_output:
                clear_output()
                print(f"\033[38;2;139;69;19m⏳ {defectdb_config['display_name']} dataset will be available soon.\033[0m")
            
            return
        
        # Check if already downloaded
        if defectdb_studio_check_dataset_exists(defectdb_new_theory):
            self.defectdb_btn_download_db.disabled = False
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f0fdf4;border-left:4px solid #86efac;border-radius:6px;font-size:12px;'>"
                f"<b>Status:</b> {defectdb_config['display_name']} dataset found in cache. Click 'Download Database' to load.</div>"
            )
            
            # Auto-load if switching between downloaded datasets
            self.defectdb_on_download_database(None)
        else:
            self.defectdb_btn_download_db.disabled = False
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
                f"<b>Status:</b> {defectdb_config['display_name']} dataset not downloaded. Click 'Download Database' to begin.</div>"
            )
    
    def defectdb_on_download_database(self, defectdb_b):
        """Download DefectDB with progress bar - SILENT ERROR HANDLING"""
        defectdb_config = defectdb_studio_THEORY_CONFIG[self.defectdb_current_theory]
        
        if not defectdb_config['available']:
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
                f"<b>Status:</b> {defectdb_config['display_name']} dataset is not available yet.</div>"
            )
            return
        
        self.defectdb_status_label.value = (
            "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
            f"<b>Status:</b> Downloading {defectdb_config['display_name']} dataset...</div>"
        )
        
        # SILENT ERROR HANDLING - No try/except error messages shown
        defectdb_db_path = defectdb_studio_download_database(
            self.defectdb_current_theory,
            defectdb_progress_widget=self.defectdb_progress_bar
        )
        
        # If download failed (None returned), update status silently
        if defectdb_db_path is None:
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
                "<b>Status:</b> Connection issue. Please try again.</div>"
            )
            return
        
        self.defectdb_progress_bar.description = "Loading:"
        self.defectdb_progress_bar.value = 50
        
        self.defectdb_df = defectdb_studio_load_csv_data(self.defectdb_current_theory)
        
        if self.defectdb_df is None:
            self.defectdb_status_label.value = (
                "<div style='padding:8px;background:#f9fafb;border-left:4px solid #6b7280;border-radius:6px;font-size:12px;'>"
                "<b>Status:</b> Data loading issue. Please try again.</div>"
            )
            return
        
        self.defectdb_progress_bar.value = 75
        self.defectdb_progress_bar.description = "Indexing:"
        
        self.defectdb_structures_index = defectdb_studio_index_structures(self.defectdb_current_theory)
        
        self.defectdb_progress_bar.value = 100
        self.defectdb_progress_bar.description = "Ready:"
        
        # Populate dropdowns
        defectdb_compounds = sorted(self.defectdb_df['AB'].unique())
        self.defectdb_data_filter_comp.options = ['All'] + defectdb_compounds
        self.defectdb_plot_comp_dropdown.options = defectdb_compounds
        self.defectdb_struct_comp.options = sorted(self.defectdb_structures_index.keys())
        
        # Show initial data
        with self.defectdb_data_output:
            clear_output()
            display(self.defectdb_df.head(20))
        
        self.defectdb_status_label.value = (
            "<div style='padding:8px;background:#f0fdf4;border-left:4px solid #86efac;border-radius:6px;font-size:12px;'>"
            f"<b>Status:</b> Ready ✓ {defectdb_config['display_name']} - "
            f"{len(self.defectdb_df)} records, {len(self.defectdb_structures_index)} compounds</div>"
        )
    
    def defectdb_on_filter_data(self, defectdb_b):
        """Filter data table"""
        if self.defectdb_df is None:
            return
        
        defectdb_filtered = self.defectdb_df.copy()
        
        if self.defectdb_data_filter_comp.value != 'All':
            defectdb_filtered = defectdb_filtered[defectdb_filtered['AB'] == self.defectdb_data_filter_comp.value]
        
        if self.defectdb_data_filter_defect.value:
            defectdb_filtered = defectdb_filtered[defectdb_filtered['Defect'].str.contains(self.defectdb_data_filter_defect.value, case=False, na=False)]
        
        with self.defectdb_data_output:
            clear_output()
            if len(defectdb_filtered) == 0:
                print("\033[38;2;139;69;19mNo results found\033[0m")
            else:
                print(f"\033[38;2;139;69;19mShowing {len(defectdb_filtered)} results:\033[0m")
                display(defectdb_filtered)
    
    def defectdb_on_export_csv(self, defectdb_b):
        """Export filtered data to CSV"""
        if self.defectdb_df is None:
            return
        
        defectdb_filtered = self.defectdb_df.copy()
        
        if self.defectdb_data_filter_comp.value != 'All':
            defectdb_filtered = defectdb_filtered[defectdb_filtered['AB'] == self.defectdb_data_filter_comp.value]
        
        if self.defectdb_data_filter_defect.value:
            defectdb_filtered = defectdb_filtered[defectdb_filtered['Defect'].str.contains(self.defectdb_data_filter_defect.value, case=False, na=False)]
        
        defectdb_csv_data = defectdb_filtered.to_csv(index=False)
        defectdb_filename = f"defectdb_{self.defectdb_current_theory.lower().replace('+', '_')}_export.csv"
        self.defectdb_data_output.clear_output(wait=True)
        with self.defectdb_data_output:
            display(defectdb_studio_create_download_link(defectdb_csv_data, defectdb_filename, "text/csv"))
    
    def defectdb_on_plot_comp_change(self, defectdb_change):
        """Update defects list when compound changes"""
        if self.defectdb_df is None or not defectdb_change.new:
            return
        
        defectdb_subset = self.defectdb_df[self.defectdb_df['AB'] == defectdb_change.new]
        defectdb_defects = sorted(defectdb_subset['Defect'].unique())
        self.defectdb_plot_defects.options = defectdb_defects
    
    def defectdb_on_generate_plot(self, defectdb_b):
        """Generate formation energy plot"""
        if self.defectdb_df is None:
            return
        
        if not self.defectdb_plot_defects.value:
            with self.defectdb_plot_output:
                clear_output()
                print("\033[38;2;139;69;19mPlease select at least one defect\033[0m")
            return
        
        # Generate plot
        defectdb_subset = self.defectdb_df[self.defectdb_df['AB'] == self.defectdb_plot_comp_dropdown.value]
        
        self.defectdb_current_plot_fig = defectdb_studio_generate_formation_energy_plot(
            defectdb_subset,
            self.defectdb_plot_comp_dropdown.value,
            self.defectdb_plot_chempot.label,
            self.defectdb_plot_chempot.value,
            self.defectdb_plot_defects.value
        )
        
        # Display ONLY in output widget
        with self.defectdb_plot_output:
            clear_output(wait=True)
            display(self.defectdb_current_plot_fig)
        
        # Close the figure to prevent automatic display
        plt.close(self.defectdb_current_plot_fig)
        
        self.defectdb_btn_download_plot.disabled = False
    
    def defectdb_on_download_plot(self, defectdb_b):
        """Download plot as PNG"""
        if self.defectdb_current_plot_fig is None:
            return
        
        defectdb_buf = io.BytesIO()
        self.defectdb_current_plot_fig.savefig(defectdb_buf, format='png', dpi=300, bbox_inches='tight', facecolor='white')
        defectdb_buf.seek(0)
        
        defectdb_comp_clean = self.defectdb_plot_comp_dropdown.value.replace('.', '_')
        defectdb_filename = f"{defectdb_comp_clean}_{self.defectdb_current_theory.lower().replace('+', '_')}_formation_energy.png"
        self.defectdb_plot_output.clear_output(wait=True)
        with self.defectdb_plot_output:
            display(defectdb_studio_create_download_link(defectdb_buf.read(), defectdb_filename, "image/png"))
    
    def defectdb_on_struct_comp_change(self, defectdb_change):
        """Update defects when compound changes"""
        if not defectdb_change.new or defectdb_change.new not in self.defectdb_structures_index:
            return
        
        defectdb_defects = sorted(self.defectdb_structures_index[defectdb_change.new].keys())
        self.defectdb_struct_defect.options = defectdb_defects
        self.defectdb_struct_defect.disabled = False
    
    def defectdb_on_struct_defect_change(self, defectdb_change):
        """Update charges when defect changes"""
        if not defectdb_change.new or not self.defectdb_struct_comp.value:
            return
        
        defectdb_comp = self.defectdb_struct_comp.value
        defectdb_defect = defectdb_change.new
        
        if defectdb_comp in self.defectdb_structures_index and defectdb_defect in self.defectdb_structures_index[defectdb_comp]:
            defectdb_charges = sorted(self.defectdb_structures_index[defectdb_comp][defectdb_defect].keys())
            self.defectdb_struct_charge.options = defectdb_charges
            self.defectdb_struct_charge.disabled = False
            self.defectdb_btn_visualize.disabled = False
    
    def defectdb_on_visualize_structure(self, defectdb_b):
        """Visualize structure"""
        if not defectdb_studio_HAS_STRUC_TOOLS:
            with self.defectdb_struct_output:
                clear_output()
                print("\033[38;2;139;69;19m❌ Structure tools not installed\033[0m")
                print("\033[38;2;139;69;19mInstall with: pip install pymatgen ase matplotlib\033[0m")
            return
        
        defectdb_comp = self.defectdb_struct_comp.value
        defectdb_defect = self.defectdb_struct_defect.value
        defectdb_charge = self.defectdb_struct_charge.value
        
        if not (defectdb_comp and defectdb_defect and defectdb_charge):
            return
        
        self.defectdb_current_structure_path = self.defectdb_structures_index[defectdb_comp][defectdb_defect][defectdb_charge]
        
        defectdb_formatted_comp = defectdb_studio_format_compound(defectdb_comp)
        defectdb_formatted_defect = defectdb_studio_format_defect(defectdb_defect)
        defectdb_title = f"{defectdb_formatted_comp} - {defectdb_formatted_defect} ({defectdb_charge})"
        
        try:
            if self.defectdb_struct_view_mode.value == 'Static Views':
                # Static visualization
                defectdb_fig, defectdb_atoms = defectdb_studio_visualize_structure_static(self.defectdb_current_structure_path, defectdb_title)
                self.defectdb_current_structure_atoms = defectdb_atoms
                self.defectdb_current_struct_fig = defectdb_fig
                
                # Display ONLY in output widget
                with self.defectdb_struct_output:
                    clear_output(wait=True)
                    display(defectdb_fig)
                
                # Close figure to prevent auto-display
                plt.close(defectdb_fig)
                
            else:
                # Interactive visualization
                defectdb_fig, defectdb_atoms = defectdb_studio_visualize_structure_matterviz(self.defectdb_current_structure_path, defectdb_title)
                self.defectdb_current_structure_atoms = defectdb_atoms
                self.defectdb_current_struct_fig = defectdb_fig
                
                # Display ONLY in output widget
                with self.defectdb_struct_output:
                    clear_output(wait=True)
                    display(defectdb_fig)
            
            # Show structure info
            defectdb_n_atoms = len(defectdb_atoms)
            defectdb_composition = defectdb_atoms.get_chemical_formula()
            
            defectdb_info_html = f"""
            <div style='background:#f9fafb;padding:12px;border-radius:8px;border-left:4px solid #6b7280;margin:10px 0;'>
                <b style='color:#6b7280;'>Structure Information:</b>
                <ul style='color:#6b7280;line-height:1.8;margin:5px 0;'>
                    <li><b>Composition:</b> {defectdb_composition}</li>
                    <li><b>Number of atoms:</b> {defectdb_n_atoms}</li>
                    <li><b>File:</b> {self.defectdb_current_structure_path.name}</li>
                </ul>
            </div>
            """
            self.defectdb_struct_info.value = defectdb_info_html
            
            # Enable download button
            self.defectdb_btn_download_structure.disabled = False
            
        except Exception as defectdb_e:
            with self.defectdb_struct_output:
                clear_output()
                print(f"\033[38;2;139;69;19m❌ Visualization failed: {defectdb_e}\033[0m")
    
    def defectdb_on_download_structure(self, defectdb_b):
        """Download structure as CIF"""
        if self.defectdb_current_structure_path is None:
            return
        
        # Always convert to CIF format
        defectdb_struct = Structure.from_file(str(self.defectdb_current_structure_path))
        defectdb_cif_str = str(CifWriter(defectdb_struct))
        defectdb_data = defectdb_cif_str.encode()
        
        defectdb_comp_clean = self.defectdb_struct_comp.value.replace('.', '_')
        defectdb_def_clean = self.defectdb_struct_defect.value.replace('+', '_')
        defectdb_filename = f"{defectdb_comp_clean}_{defectdb_def_clean}_{self.defectdb_struct_charge.value}.cif"
        self.defectdb_struct_output.clear_output(wait=True)
        with self.defectdb_struct_output:
            display(defectdb_studio_create_download_link(defectdb_data, defectdb_filename, "chemical/x-cif"))

# =============================================================================
# RUN APP
# =============================================================================

defectdb_studio_app = DefectDBStudioApp()
defectdb_studio_app.defectdb_display()